In [1]:
import pandas as pd
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [2]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
df = pd.read_csv("quora_cleaned_sample.csv")

df.head()

,doc_id,original_text,cleaned_text
0,1,What is the step by step guide to invest in sh...,step step guide invest share market india
1,2,What is the step by step guide to invest in sh...,step step guide invest share market
2,3,What is the story of Kohinoor (Koh-i-Noor) Dia...,story kohinoor diamond
3,4,What would happen if the Indian government sto...,would happen indian government stole kohinoor ...
4,5,How can I increase the speed of my internet co...,increase speed internet connection using vpn


In [4]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

print(len(stop_words))

198


In [5]:
from nltk.tokenize import word_tokenize
import string

def process_query(query):

    tokens = word_tokenize(query)

    filtered_tokens = [
        word.lower()
        for word in tokens
        if word.lower() not in stop_words
        and word not in string.punctuation
    ]

    return filtered_tokens

In [6]:
query = "How can I learn machine learning quickly?"

print(process_query(query))

['learn', 'machine', 'learning', 'quickly']


In [7]:
def query_service(query):

    processed_query = process_query(query)

    return {
        "original_query": query,
        "processed_query": processed_query
    }

In [8]:
result = query_service("How can I learn machine learning quickly?")

print(result)

{'original_query': 'How can I learn machine learning quickly?', 'processed_query': ['learn', 'machine', 'learning', 'quickly']}


In [9]:
queries = [
    "What is Artificial Intelligence?",
    "How to learn Python fast?",
    "Best books for Data Science"
]

for q in queries:
    print(query_service(q))
    print("-" * 50)

{'original_query': 'What is Artificial Intelligence?', 'processed_query': ['artificial', 'intelligence']}
--------------------------------------------------
{'original_query': 'How to learn Python fast?', 'processed_query': ['learn', 'python', 'fast']}
--------------------------------------------------
{'original_query': 'Best books for Data Science', 'processed_query': ['best', 'books', 'data', 'science']}
--------------------------------------------------


In [10]:
from nltk.corpus import wordnet

nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [11]:
def get_synonyms(word):

    synonyms = set()

    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ")

            if synonym.lower() != word.lower():
                synonyms.add(synonym.lower())

    return list(synonyms)

In [12]:
print(get_synonyms("car")[:10])

['railway car', 'motorcar', 'elevator car', 'auto', 'railroad car', 'gondola', 'cable car', 'railcar', 'machine', 'automobile']


In [13]:
def expand_query(query):

    processed_query = process_query(query)

    expanded_terms = set(processed_query)

    for word in processed_query:

        synonyms = get_synonyms(word)

        expanded_terms.update(synonyms[:3])

    return list(expanded_terms)

In [14]:
print(expand_query("car insurance"))

['insurance', 'railway car', 'motorcar', 'elevator car', 'policy', 'insurance policy', 'indemnity', 'car']


In [15]:
!pip install textblob

   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/625.0 kB ? eta -:--:--
   ---------------------------------------- 625.0/625.0 kB 946.9 kB/s  0:00:01


In [16]:
from textblob import TextBlob

In [17]:
def correct_query(query):

    corrected_query = str(TextBlob(query).correct())

    return corrected_query

In [18]:
print(correct_query("machne learnng"))

machine learning


In [19]:
def query_service_v3(query):

    # 1. Spell Correction
    corrected_query = str(TextBlob(query).correct())

    # 2. Processing (tokenization + stopwords)
    processed_query = process_query(corrected_query)

    # 3. Synonym Expansion
    expanded_query = expand_query(corrected_query)

    return {
        "original_query": query,
        "corrected_query": corrected_query,
        "processed_query": processed_query,
        "expanded_query": expanded_query
    }

In [20]:
result = query_service_v3("machne learnng")

print(result)

{'original_query': 'machne learnng', 'corrected_query': 'machine learning', 'processed_query': ['machine', 'learning'], 'expanded_query': ['get word', 'motorcar', 'memorize', 'erudition', 'learning', 'political machine', 'car', 'machine']}


In [21]:
def get_synonyms(word):

    synonyms = set()

    for syn in wordnet.synsets(word):

        # خذ أول معنى فقط (يحسن الجودة)
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ")

            if synonym.lower() != word.lower():
                synonyms.add(synonym.lower())

        break  # 👈 هذا أهم تعديل

    return list(synonyms)

In [22]:
print(get_synonyms("machine"))

[]


In [23]:
def get_synonyms(word):

    synonyms = set()

    synsets = wordnet.synsets(word)[:3]  # 👈 أهم تعديل

    for syn in synsets:
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ")

            if synonym.lower() != word.lower():
                synonyms.add(synonym.lower())

    return list(synonyms)

In [24]:
print(get_synonyms("machine"))
print(get_synonyms("car"))
print(get_synonyms("learning"))

[]
['railway car', 'motorcar', 'auto', 'railroad car', 'gondola', 'railcar', 'machine', 'automobile']
['learn', 'learnedness', 'erudition', 'eruditeness', 'encyclopaedism', 'acquire', 'scholarship', 'acquisition', 'encyclopedism', 'larn']


In [25]:
def get_synonyms(word):

    synonyms = set()

    synsets = wordnet.synsets(word)[:5]

    for syn in synsets:
        for lemma in syn.lemmas():

            synonym = lemma.name().replace("_", " ").lower()

            # ❌ استبعاد نفس الكلمة
            if synonym == word.lower():
                continue

            # ❌ استبعاد الكلمات الطويلة الغريبة
            if len(synonym.split()) > 2:
                continue

            # ❌ استبعاد الكلمات النادرة جدًا (اختياري بسيط)
            if synonym.isalpha():
                synonyms.add(synonym)

    return list(synonyms)

In [26]:
print(get_synonyms("machine"))
print(get_synonyms("car"))
print(get_synonyms("learning"))

[]
['motorcar', 'auto', 'gondola', 'railcar', 'machine', 'automobile']
['learn', 'hear', 'discover', 'learnedness', 'erudition', 'memorize', 'eruditeness', 'encyclopaedism', 'acquire', 'see', 'scholarship', 'acquisition', 'memorise', 'encyclopedism', 'larn', 'con']


In [27]:
def get_synonyms(word):

    synonyms = set()

    synsets = wordnet.synsets(word)

    if not synsets:
        return []

    syn = synsets[0]

    for lemma in syn.lemmas():

        synonym = lemma.name().replace("_", " ").lower()

        if synonym == word.lower():
            continue

        if not synonym.isalpha():
            continue

        if len(synonym) < 3:
            continue

        synonyms.add(synonym)

    return list(synonyms)

In [28]:
print(get_synonyms("machine"))
print(get_synonyms("car"))
print(get_synonyms("learning"))

[]
['machine', 'motorcar', 'automobile', 'auto']
['acquisition']


In [29]:
def get_synonyms(word):

    synonyms = set()

    synsets = wordnet.synsets(word)[:2]  # 👈 توازن مهم

    for syn in synsets:

        for lemma in syn.lemmas():

            synonym = lemma.name().replace("_", " ").lower()

            if synonym == word.lower():
                continue

            if not synonym.isalpha():
                continue

            if len(synonym) < 3:
                continue

            synonyms.add(synonym)

    return list(synonyms)

In [30]:
print(get_synonyms("machine"))
print(get_synonyms("learning"))

[]
['learnedness', 'erudition', 'eruditeness', 'encyclopaedism', 'scholarship', 'acquisition', 'encyclopedism']


In [31]:
import re

def process_query_service(query, expand=True):

    query_clean = re.sub(r'[^a-z\s]', '', query.lower())
    words = query_clean.split()

    if not expand:
        return " ".join(words)

    local_syns = {
        "reduce": ["lose", "decrease"],
        "weight": ["fat", "body"],
        "internet": ["net", "web"],
        "invest": ["money", "market"]
    }

    expanded_words = list(words)

    for w in words:
        if w in local_syns:
            for syn in local_syns[w]:
                if syn not in expanded_words:
                    expanded_words.append(syn)

    return " ".join(expanded_words)

In [32]:
query = "reduce weight safely"

result = process_query_service(query)

print("Original:", query)
print("Expanded:", result)

Original: reduce weight safely
Expanded: reduce weight safely lose decrease fat body


In [33]:
query = "how to reduce weight safely"
print("Original:", query)
print("Expanded:", process_query_service(query))

Original: how to reduce weight safely
Expanded: how to reduce weight safely lose decrease fat body


In [34]:
!jupyter nbconvert --to script ghazalsally.ipynb

[NbConvertApp] Converting notebook ghazalsally.ipynb to script
[NbConvertApp] Writing 6550 bytes to ghazalsally.py
